In [ ]:
import glob
import re
import matplotlib.pyplot as plt
import os

# ── Konfiguration ──────────────────────────────────────────────────────────────
output_folder  = "./results"
figures_folder = "./figures"

EXTRA_BASELINES = [
    (
        "Baseline (mgk_g4_d4 original)",
        os.path.expanduser("~/speechbrain/results/hparams/train_mgk_g4_d4"),
    ),
    (
        "Baseline (mgk_g4_d4 modified lwf)",
        os.path.expanduser("~/speechbrain/MetricGAN-KAN-LwF/results_2603/hparams/train_mgk_g4_d4"),
    ),
]
# ──────────────────────────────────────────────────────────────────────────────

COLORS  = ["#1b9e77","#d95f02","#7570b3","#e7298a",
           "#66a61e","#e6ab02","#a6761d","#666666"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

PLOTS = [
    ("valid_pesq", "Valid PESQ"),
    ("valid_csig", "Valid CSIG"),
    ("valid_cbak", "Valid CBAK"),
    ("valid_covl", "Valid COVL"),
]


# ── Hilfsfunktionen ────────────────────────────────────────────────────────────

def _extract_metric(line, keys):
    for key in keys:
        m = re.search(
            rf"{key}\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
            line, flags=re.IGNORECASE,
        )
        if m:
            return float(m.group(1))
    return None


def read_hparams(log_file):
    """Liest hyperparams.yaml im selben Ordner wie log_file."""
    yaml_path = os.path.join(os.path.dirname(log_file), "hyperparams.yaml")
    pd_lambda = pd_interval = None
    if os.path.isfile(yaml_path):
        with open(yaml_path, "r", encoding="utf-8") as f:
            for raw_line in f:
                line = raw_line.rstrip()
                if pd_lambda is None:
                    m = re.match(r"\s*pd_lambda\s*:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
                    if m: pd_lambda = m.group(1)
                if pd_interval is None:
                    m = re.match(r"\s*pd_anchor_interval\s*:\s*(\d+)", line)
                    if m: pd_interval = int(m.group(1))
    parts = []
    if pd_lambda   is not None: parts.append(f"\u03bb={pd_lambda}")
    if pd_interval is not None: parts.append(f"I={pd_interval}")
    return {
        "pd_lambda":          pd_lambda,
        "pd_anchor_interval": pd_interval,
        "label_suffix":       f" ({', '.join(parts)})" if parts else "",
    }


def load_training_curve(log_file):
    """Liest train_log.txt und hyperparams.yaml; gibt vollstaendiges curves-Dict."""
    epochs, valid_pesq, valid_csig, valid_cbak, valid_covl = [], [], [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if "epoch" not in line.lower(): continue
            m_epoch = re.search(r"epoch\s*[:=]\s*(\d+)", line, flags=re.IGNORECASE)
            if not m_epoch: continue
            epochs.append(int(m_epoch.group(1)))
            valid_pesq.append(_extract_metric(line, [r"pesq", r"valid[_\s-]*pesq"]))
            valid_csig.append(_extract_metric(line, [r"csig", r"valid[_\s-]*csig"]))
            valid_cbak.append(_extract_metric(line, [r"cbak", r"valid[_\s-]*cbak"]))
            valid_covl.append(_extract_metric(line, [r"covl", r"valid[_\s-]*covl"]))
    hparams = read_hparams(log_file)
    return {
        "epoch":              epochs,
        "valid_pesq":         valid_pesq,
        "valid_csig":         valid_csig,
        "valid_cbak":         valid_cbak,
        "valid_covl":         valid_covl,
        "pd_anchor_interval": hparams["pd_anchor_interval"],
        "label_suffix":       hparams["label_suffix"],
    }


def resolve_log_file(path):
    if os.path.isfile(path): return path
    candidate = os.path.join(path, "train_log.txt")
    if os.path.isfile(candidate): return candidate
    hits = glob.glob(os.path.join(path, "**/train_log.txt"), recursive=True)
    return sorted(hits)[0] if hits else None


def draw_anchor_vlines(ax, curves, show_label=True):
    """Rote gestrichelte Linien bei jedem Vielfachen von pd_anchor_interval."""
    interval = curves.get("pd_anchor_interval")
    epochs   = curves.get("epoch", [])
    if not interval or not epochs:
        return
    first = True
    for epoch in range(interval, max(epochs) + 1, interval):
        ax.axvline(
            epoch,
            color="red", linewidth=0.9, linestyle="--", alpha=0.55, zorder=1,
            label=f"anchor (I={interval})" if (first and show_label) else "_nolegend_",
        )
        first = False


# ── Runs laden ─────────────────────────────────────────────────────────────────
os.makedirs(figures_folder, exist_ok=True)

all_curves      = {}
baseline_labels = set()

candidates = sorted(glob.glob(f"{output_folder}/**/train_log.txt", recursive=True))
if not candidates:
    print(f"[INFO] Kein train_log.txt in '{output_folder}' gefunden.")

for log_file in candidates:
    curves    = load_training_curve(log_file)
    base_name = os.path.relpath(os.path.dirname(log_file), output_folder)
    run_label = base_name + curves["label_suffix"]
    if not curves["epoch"]:
        print(f"[WARN] Keine Epochenzeilen in {log_file} – uebersprungen.")
        continue
    all_curves[run_label] = curves

for label, path in EXTRA_BASELINES:
    log_file = resolve_log_file(path)
    if log_file is None:
        print(f"[WARN] Keine train_log.txt fuer '{label}' ({path}) – uebersprungen.")
        continue
    curves     = load_training_curve(log_file)
    if not curves["epoch"]:
        print(f"[WARN] Keine Epochenzeilen in {log_file} – uebersprungen.")
        continue
    full_label = label + curves["label_suffix"]
    all_curves[full_label] = curves
    baseline_labels.add(full_label)
    print(f"[INFO] Baseline geladen: '{full_label}' ({log_file})")

if not all_curves:
    raise RuntimeError("Keine gueltigen Runs gefunden.")


# ── 3) Pro Run: 2x2-Diagramm ──────────────────────────────────────────────────
for run_label, curves in all_curves.items():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
    for ax, (key, title) in zip(axes.flat, PLOTS):
        y      = curves[key]
        x      = [e for e, v in zip(curves["epoch"], y) if v is not None]
        y_vals = [v for v in y if v is not None]
        if x:
            ax.plot(x, y_vals, marker="o", linewidth=1.5, markersize=3, zorder=2)
        draw_anchor_vlines(ax, curves)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)
        if curves["pd_anchor_interval"]:
            ax.legend(fontsize=7, loc="lower right")

    fig.suptitle(f"Training Curves \u2013 {run_label}", fontsize=12)
    fig.tight_layout()
    safe_label = re.sub(r"[\\/:*?\"<>|()\s,=]", "_", run_label)
    out_png = os.path.join(figures_folder, f"training_curves_{safe_label}.png")
    fig.savefig(out_png, bbox_inches="tight")
    plt.show()
    print(f"Diagramm gespeichert: {out_png}")


# ── 4) Kombiniertes PESQ-Diagramm ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6), dpi=120)

color_cycle     = iter(COLORS  * 10)
marker_cycle    = iter(MARKERS * 10)
drawn_intervals = set()

for run_label, curves in all_curves.items():
    is_baseline = run_label in baseline_labels
    y      = curves["valid_pesq"]
    x      = [e for e, v in zip(curves["epoch"], y) if v is not None]
    y_vals = [v for v in y if v is not None]
    if not x: continue

    color  = next(color_cycle)
    marker = next(marker_cycle)

    ax.plot(
        x, y_vals,
        color=color, marker=marker,
        linewidth=2.5 if is_baseline else 1.5,
        markersize=5  if is_baseline else 3,
        linestyle=(0, (6, 2)) if is_baseline else "-",
        alpha=0.9, zorder=3 if is_baseline else 2,
        label=run_label,
    )

    # End-Label: Runname + Parameter als zweite Zeile
    suffix    = curves["label_suffix"]
    base_name = run_label[: len(run_label) - len(suffix)]
    param_str = suffix.strip(" ()")
    ann_text  = base_name if not param_str else f"{base_name}\n{param_str}"
    ax.annotate(
        ann_text,
        xy=(x[-1], y_vals[-1]),
        xytext=(5, 0), textcoords="offset points",
        fontsize=7.5, color=color, va="center",
        multialignment="left",
        fontweight="bold" if is_baseline else "normal",
    )



ax.set_title("Valid PESQ \u2013 alle Runs", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("PESQ")
ax.grid(True, alpha=0.25, linestyle=":")

x_max_all = max(
    max(e for e, v in zip(c["epoch"], c["valid_pesq"]) if v is not None)
    for c in all_curves.values() if any(v is not None for v in c["valid_pesq"])
)
for e in range(50, int(x_max_all) + 1, 50):
    ax.axvline(e, color="gray", linewidth=0.6, linestyle=":", alpha=0.5, zorder=0)
ax.set_xlim(right=x_max_all * 1.28)
ax.legend(fontsize=7.5, loc="lower right", framealpha=0.75, title="Runs")
fig.tight_layout()

out_combined = os.path.join(figures_folder, "training_curves_pesq_combined.png")
fig.savefig(out_combined, bbox_inches="tight")
plt.show()
print(f"Kombiniertes PESQ-Diagramm gespeichert: {out_combined}")